# Phase 1 — Data Understanding and the leakage check

[`walkthrough/phase1.md`](../walkthrough/phase1.md). Reads `dataset/households.parquet` and
`dataset/features.parquet` (both built by `uv run sgc build --tsv ...`). All logic lives in
`src/savings_goal`; this notebook only calls it and prints aggregates.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

from savings_goal import config as C
from savings_goal import viz
from savings_goal.data.build import reconcile_totals
from savings_goal.evaluation import leakage
from savings_goal.evaluation.metrics import weighted_mean
from savings_goal.features.engineer import spec_from_frame
from savings_goal.io import load_features, load_households, write_result
from savings_goal.models.pipeline import xgb_pipeline

pd.set_option("display.max_columns", 60)
viz.use()

df = load_households()
feats = load_features()
spec = spec_from_frame(feats)
SHARE_COLS, EXPENSE = C.SHARE_COLS, C.EXPENSE_CATEGORIES
BEHAVIOUR_COLS = [c for c in df.columns if c.startswith("Has_")]
print(f"households: {len(df):,}   columns: {df.shape[1]}   PSUs: {df[C.GROUP_COL].nunique():,}")

households: 41,518   columns: 54   PSUs: 2,461


## Q1 — What does each column mean, and what unit/time period does it represent?

In [2]:
ID_COLS = ["STATEID", "DISTID", "PSUID", "IDPSU", "HHID", "HHSPLITID", "IDHH"]
roles = []
for c in df.columns:
    if c in ID_COLS:
        role, unit = "Identifier", "survey code"
    elif c == "WT":
        role, unit = "Survey weight", "households represented"
    elif c in C.LEAKAGE_COLS:
        role, unit = "LEAKAGE - excluded", "proportion" if c == "Savings_Rate" else "annual Rs"
    elif c == "Goal_Met":
        role, unit = "TARGET", "binary"
    elif c in BEHAVIOUR_COLS:
        role, unit = "External validation - excluded", "binary"
    elif c in SHARE_COLS:
        role, unit = "Feature", "share of total expenditure"
    elif c == "Occupation_Tie":
        role, unit = "Build diagnostic", "binary"
    elif c == "INCOME":
        role, unit = "Feature (as Log_Income)", "annual Rs"
    elif df[c].dtype == object:
        role, unit = "Feature", "categorical"
    else:
        role, unit = "Feature", "count / ratio"
    roles.append({"column": c, "role": role, "unit": unit, "dtype": str(df[c].dtype)})
inventory = pd.DataFrame(roles)
print(inventory.groupby("role").size().to_string())

role
Build diagnostic                   1
External validation - excluded     6
Feature                           22
Feature (as Log_Income)            1
Identifier                         7
LEAKAGE - excluded                15
Survey weight                      1
TARGET                             1


## Q2 — Distribution of income, expenses, and savings

In [3]:
money = ["INCOME", "COTOTAL", "Savings", *EXPENSE]
desc = df[money].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
desc["skew"] = df[money].skew()
print("Money columns (annual Rs):")
print(desc.round(1).to_string())
print("\nSavings rate:")
print(df["Savings_Rate"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(4).to_string())

checks = {
    "consumption exceeds income (negative savings)": df["Savings"] < 0,
    "savings rate below -100% (spends >2x income)": df["Savings_Rate"] < -1,
    "zero rent recorded": df["Rent"] == 0,
    "head age below 18": df["Head_Age"] < 18,
    "debt exceeds 10x annual income": df["Debt_To_Income"] > 10,
    "debt (DB5) missing": df["Debt_Missing"] == 1,
    "occupation tie broken by priority": df["Occupation_Tie"] == 1,
}
print("\n=== Implausible / structurally awkward values (unweighted, then survey-weighted) ===")
for label, mask in checks.items():
    print(f"  {label:46s} {mask.sum():6,}  ({mask.mean():6.2%})  weighted {weighted_mean(mask, df['WT']):6.2%}")

Money columns (annual Rs):
                       mean       std        min      25%      50%       75%         max   skew
INCOME             129977.4  217306.4      100.0  40230.0  75000.0  145717.5  11360000.0   15.8
COTOTAL            117938.8  120992.8      180.0  56251.0  87175.0  139984.5   4080760.0    7.2
Savings             12038.6  196600.3 -4000836.0 -38619.8  -6487.0   33367.0  11083796.0   15.4
Groceries           44770.5   26549.4        0.0  27360.0  39612.0   55716.0    764880.0    2.6
Eating_Out           1141.3    3772.4        0.0      0.0      0.0     720.0    240000.0   14.8
Utilities           10209.8   14095.3        0.0   4896.0   8376.0   12852.0   1079016.0   48.0
Rent                 1530.0   10124.1        0.0      0.0      0.0       0.0    720000.0   24.3
Transport           10016.1   25966.9        0.0   1200.0   3600.0   10800.0   1228800.0   17.8
Healthcare          12125.4   38845.1        0.0    822.5   3600.0   11200.0   4001800.0   35.4
Education    

## Q3 — Missing values and duplicate rows

In [4]:
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
print(pd.DataFrame({"n_missing": miss, "pct": (miss / len(df) * 100).round(3)}).to_string())
print("\nDuplicate household identifiers:", df["IDHH"].duplicated().sum())

                         n_missing    pct
Debt_To_Income                3150  7.587
Has_Gold_Jewellery             176  0.424
Has_Pension_LIC                155  0.373
Has_Post_Office_Account        142  0.342
Has_Fixed_Deposit              138  0.332
Has_Bank_Savings               133  0.320
Has_Securities                 133  0.320
Caste_Group                     85  0.205
Max_Adult_Education              6  0.014

Duplicate household identifiers: 0


## Q4 — How correlated are expense categories with income and with each other?

In [5]:
raw_corr = df[EXPENSE].corrwith(df["INCOME"], method="spearman").sort_values(ascending=False)
print("Spearman: raw rupee categories vs INCOME (the leaky representation)")
print(raw_corr.round(4).to_string())
share_corr = df[SHARE_COLS].corrwith(df["Log_Income"], method="spearman").sort_values()
print("\nSpearman: composition shares vs income")
print(share_corr.round(4).to_string())

share_cm = df[SHARE_COLS].corr(method="spearman")
labels = [c.replace("_Share", "").replace("_", " ") for c in SHARE_COLS]
mask = np.triu(np.ones_like(share_cm, dtype=bool))
shown = share_cm.mask(mask)
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(shown, cmap=viz.DIVERGING, vmin=-0.5, vmax=0.5)
for i in range(len(labels)):
    for j in range(i):
        v = share_cm.iat[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                color="#ffffff" if abs(v) > 0.35 else viz.INK_2)
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
for side in ("left", "bottom"):
    ax.spines[side].set_visible(False)
ax.tick_params(length=0)
cb = fig.colorbar(im, ax=ax, shrink=0.7, label="Spearman rho")
cb.outline.set_visible(False)
off = share_cm.where(~np.eye(len(SHARE_COLS), dtype=bool)).stack()
ax.set_title(f"Correlation between expense shares: {(off < 0).mean():.0%} of pairs are negative,\n"
             "as expected for parts of a whole that sum to one")
plt.savefig(C.RESULTS / "phase1_share_correlations.png")
plt.show()

Spearman: raw rupee categories vs INCOME (the leaky representation)


Groceries            0.5811
Utilities            0.5473
Transport            0.5040
Miscellaneous        0.4724
Clothing_Footwear    0.4366
Insurance            0.3582
Education            0.2806
Entertainment        0.2500
Eating_Out           0.2193
Rent                 0.1128
Healthcare           0.0859



Spearman: composition shares vs income
Groceries_Share           -0.2800
Healthcare_Share          -0.1222
Utilities_Share           -0.0345
Clothing_Footwear_Share   -0.0332
Miscellaneous_Share        0.0088
Rent_Share                 0.1094
Eating_Out_Share           0.1827
Education_Share            0.1928
Entertainment_Share        0.2203
Transport_Share            0.3284
Insurance_Share            0.3315


## Q5 — Leakage: is the target derivable from any candidate feature?

Univariate checks first (tests A, B, C), then the multivariate check: can the
*combination* of household size, the food share and income recover total spend?

In [6]:
rec = reconcile_totals(__import__("polars").from_pandas(df))
print("=== COTOTAL (target) vs the sum of our 11 categories (shares) ===")
print(f"  Pearson r {rec['pearson_r']:.4f}; median difference Rs {rec['median_diff']:,.0f}; "
      f"within 1%: {rec['within_1pct']:.2%}; within 5%: {rec['within_5pct']:.2%}")
print(f"  label recomputed from the category sum agrees with Goal_Met for {rec['label_agreement']:.2%}")
print("  -> Savings is an exact identity over INCOME and COTOTAL, and only approximately")
print("     over the expense categories (they reproduce COTOTAL within 1% for most households).")

print("\n=== Test A: raw categories + INCOME reconstruct the target? ===")
implied_raw = ((df["INCOME"] - df[EXPENSE].sum(axis=1)) / df["INCOME"] >= 0.20).astype(int)
agree_raw = (implied_raw == df["Goal_Met"]).mean()
print(f"  agreement with Goal_Met: {agree_raw:.4f}  -> EXCLUDE raw categories")

print("\n=== Test B: expense-to-INCOME ratios reconstruct the target? ===")
implied_ratio = (df[EXPENSE].div(df["INCOME"], axis=0).sum(axis=1) <= 0.80).astype(int)
agree_ratio = (implied_ratio == df["Goal_Met"]).mean()
print(f"  agreement with Goal_Met: {agree_ratio:.4f}  -> EXCLUDE income ratios")

print("\n=== Test C: does any single share carry the target? (Spearman + marginal AUC) ===")
rows = []
for c in SHARE_COLS:
    rho, _ = spearmanr(df[c], df["Savings_Rate"])
    auc = roc_auc_score(df["Goal_Met"], df[c])
    rows.append({"share": c, "Spearman vs Savings_Rate": rho, "marginal ROC-AUC": auc,
                 "|AUC - 0.5|": abs(auc - 0.5)})
test_c = pd.DataFrame(rows).set_index("share").sort_values("|AUC - 0.5|", ascending=False)
print(test_c.round(4).to_string())
print("\nNo single share is a strong marginal predictor. That is NOT the same as saying the")
print("shares are leakage-free: the checks below test them jointly with size and income.")

=== COTOTAL (target) vs the sum of our 11 categories (shares) ===
  Pearson r 0.9932; median difference Rs 0; within 1%: 97.72%; within 5%: 98.29%
  label recomputed from the category sum agrees with Goal_Met for 99.75%
  -> Savings is an exact identity over INCOME and COTOTAL, and only approximately
     over the expense categories (they reproduce COTOTAL within 1% for most households).

=== Test A: raw categories + INCOME reconstruct the target? ===
  agreement with Goal_Met: 0.9975  -> EXCLUDE raw categories

=== Test B: expense-to-INCOME ratios reconstruct the target? ===
  agreement with Goal_Met: 0.9975  -> EXCLUDE income ratios

=== Test C: does any single share carry the target? (Spearman + marginal AUC) ===


                         Spearman vs Savings_Rate  marginal ROC-AUC  |AUC - 0.5|
share                                                                           
Healthcare_Share                          -0.2090            0.3997       0.1003
Utilities_Share                            0.1815            0.5810       0.0810
Transport_Share                            0.1172            0.5628       0.0628
Insurance_Share                            0.1153            0.5514       0.0514
Groceries_Share                            0.1381            0.5436       0.0436
Clothing_Footwear_Share                    0.0974            0.5356       0.0356
Education_Share                           -0.0477            0.4681       0.0319
Eating_Out_Share                           0.0636            0.5254       0.0254
Entertainment_Share                        0.0370            0.5160       0.0160
Miscellaneous_Share                       -0.0212            0.4915       0.0085
Rent_Share                  

In [7]:
print("=== Food-size fit: are food rupees a function of household size? ===")
food_fit = leakage.food_size_fit(df)
print(f"  R^2[log(Groceries) ~ log(Household_Size)] = {food_fit['r2']:.4f}  "
      f"(elasticity {food_fit['elasticity']:.3f}, n = {food_fit['n']:,})")

print("\n=== Size x food-share oracle (no model) ===")
print("  implied_sr = 1 - (k * Household_Size / Groceries_Share) / INCOME,  k = median food Rs per person")
oracle = leakage.size_share_oracle(df)
print(f"  k = Rs {oracle.k:,.0f} per person per year")
print(f"  excluded {oracle.n_excluded} households with Groceries_Share == 0 or NaN "
      f"({oracle.n_excluded / len(df):.3%}); scored {oracle.n_used:,}")
print(f"  ROC-AUC(Goal_Met, implied_sr) = {oracle.auc:.4f}   (no learning, no CV: three columns and a median)")
print(f"  label agreement if implied_sr >= 0.20: {oracle.agreement_at_threshold:.4f}")

=== Food-size fit: are food rupees a function of household size? ===
  R^2[log(Groceries) ~ log(Household_Size)] = 0.2920  (elasticity 0.630, n = 41,494)

=== Size x food-share oracle (no model) ===
  implied_sr = 1 - (k * Household_Size / Groceries_Share) / INCOME,  k = median food Rs per person
  k = Rs 8,848 per person per year
  excluded 24 households with Groceries_Share == 0 or NaN (0.058%); scored 41,494
  ROC-AUC(Goal_Met, implied_sr) = 0.8950   (no learning, no CV: three columns and a median)
  label agreement if implied_sr >= 0.20: 0.8218


In [8]:
print("=== Ablation under grouped 5-fold CV (XGBoost, untuned default config) ===")
sets = leakage.ablation_sets(spec.features, spec.indicators, SHARE_COLS)


def make(cols):
    return xgb_pipeline([c for c in cols if c not in C.CATEGORICALS],
                        [c for c in cols if c in C.CATEGORICALS])


ablation = leakage.ablation(feats, feats["Goal_Met"], feats[C.GROUP_COL], sets, make)
print(ablation.round(4).to_string())

=== Ablation under grouped 5-fold CV (XGBoost, untuned default config) ===


                                      n features  ROC-AUC  ROC-AUC sd  F1 (macro)  PR-AUC (at-risk)  AUC retained
feature set                                                                                                      
all features                                  29   0.9312      0.0034      0.8359            0.9665        1.0000
no Groceries_Share                            28   0.9291      0.0032      0.8339            0.9655        0.9977
no size family                                26   0.9213      0.0028      0.8256            0.9612        0.9893
no shares or indicators (deployable)          13   0.8788      0.0062      0.7753            0.9402        0.9437


In [9]:
no_shares = ablation.loc["no shares or indicators (deployable)"]
shares_diagnostic = (oracle.auc >= 0.85) or (no_shares["AUC retained"] >= 0.90)
print("=== Decision: which feature set is the headline? ===")
print(f"  oracle AUC {oracle.auc:.3f} {'>=' if oracle.auc >= 0.85 else '<'} 0.85;  "
      f"no-shares model retains {no_shares['AUC retained']:.1%} of the full model's AUC")
if shares_diagnostic:
    print("  -> The headline model uses income + demographics + debt (the deployable set).")
    print("     Spending shares are diagnostic: with size and income they reconstruct total spend,")
    print("     so their extra skill is largely a restatement of the label.")
else:
    print("  -> Shares carry signal beyond the size/share route; keep the full model as headline.")

write_result("leakage", {
    "test_a_raw_agreement": agree_raw,
    "test_b_ratio_agreement": agree_ratio,
    "test_c": test_c.reset_index(),
    "reconcile_totals": rec,
    "ablation": ablation.reset_index(),
    "food_size_fit": food_fit,
    "oracle": {"auc": oracle.auc, "k": oracle.k, "n_used": oracle.n_used, "n_excluded": oracle.n_excluded,
           "agreement_at_threshold": oracle.agreement_at_threshold},
    "decision": {"shares_diagnostic": shares_diagnostic,
                 "headline": "deployable" if shares_diagnostic else "full"},
})
print("\nwrote results/leakage.json")

=== Decision: which feature set is the headline? ===
  oracle AUC 0.895 >= 0.85;  no-shares model retains 94.4% of the full model's AUC
  -> The headline model uses income + demographics + debt (the deployable set).
     Spending shares are diagnostic: with size and income they reconstruct total spend,
     so their extra skill is largely a restatement of the label.

wrote results/leakage.json


## Q6 — Class balance, unweighted and survey-weighted

In [10]:
w = df["WT"]
vc = df["Goal_Met"].value_counts().sort_index()
rate, rate_w = df["Goal_Met"].mean(), weighted_mean(df["Goal_Met"], w)
print(pd.DataFrame({"n": vc, "pct": (vc / len(df) * 100).round(2)}).to_string())
print(f"\nimbalance ratio: {vc.max() / vc.min():.2f} : 1")
print(f"Goal_Met rate: unweighted {rate:.4f}   survey-weighted {rate_w:.4f}")

sens = []
for t in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]:
    g = (df["Savings_Rate"] >= t).astype(int)
    sens.append({"threshold": t, "Goal_Met rate": g.mean(), "weighted rate": weighted_mean(g, w),
                 "agreement with 20% label": (g == df["Goal_Met"]).mean()})
sens = pd.DataFrame(sens)
print("\nSensitivity to the (conventional) threshold:")
print(sens.round(4).to_string(index=False))

by_area = df.groupby("Area_Type").apply(
    lambda g: pd.Series({"rate": g["Goal_Met"].mean(), "weighted": weighted_mean(g["Goal_Met"], g["WT"]),
                         "n": len(g)}))
print("\nGoal_Met by area type:")
print(by_area.sort_values("rate", ascending=False).round(4).to_string())

write_result("phase1_eda", {
    "n_households": len(df), "n_psu": df[C.GROUP_COL].nunique(),
    "goal_met_rate": rate, "goal_met_rate_weighted": rate_w,
    "consumption_exceeds_income": float((df["Savings"] < 0).mean()),
    "consumption_exceeds_income_weighted": weighted_mean(df["Savings"] < 0, w),
    "spends_over_2x_income": float((df["Savings_Rate"] < -1).mean()),
    "threshold_sensitivity": sens, "by_area": by_area.reset_index(),
    "has_fixed_deposit_by_goal": df.groupby("Goal_Met")["Has_Fixed_Deposit"].mean(),
})
print("wrote results/phase1_eda.json")

              n    pct
Goal_Met              
0         28262  68.07
1         13256  31.93

imbalance ratio: 2.13 : 1
Goal_Met rate: unweighted 0.3193   survey-weighted 0.3047

Sensitivity to the (conventional) threshold:
 threshold  Goal_Met rate  weighted rate  agreement with 20% label
      0.00         0.4411         0.4231                    0.8782
      0.05         0.4138         0.3966                    0.9054
      0.10         0.3833         0.3669                    0.9360
      0.15         0.3505         0.3353                    0.9687
      0.20         0.3193         0.3047                    1.0000
      0.25         0.2868         0.2740                    0.9676
      0.30         0.2532         0.2396                    0.9340
      0.40         0.1876         0.1757                    0.8683

Goal_Met by area type:
                          rate  weighted        n
Area_Type                                        
Metro_Urban             0.4212    0.4317   3063.0


## Summary figure

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.subplots_adjust(hspace=0.45, wspace=0.45)

ax = axes[0, 0]
ax.hist(df["Log_Income"], bins=60, color=viz.BLUE, edgecolor=viz.SURFACE, linewidth=0.5)
viz.strip(ax, "y")
ax.set_xlabel("log annual income (Rs)")
ax.set_ylabel("households")
ax.set_title(f"Income: raw skew {df['INCOME'].skew():.1f}, close to symmetric after the log")

ax = axes[0, 1]
ax.hist(df["Savings_Rate"].clip(-2, 1), bins=60, color=viz.CONTEXT, edgecolor=viz.SURFACE, linewidth=0.5)
ax.axvline(0.20, color=viz.BLUE, lw=1.5)
ax.annotate("20% benchmark", (0.20, ax.get_ylim()[1] * 0.92), xytext=(-6, 0), textcoords="offset points",
            ha="right", color=viz.INK_2)
viz.strip(ax, "y")
ax.set_xlabel("savings rate (clipped to [-2, 1])")
ax.set_title(f"Savings rate: median {df['Savings_Rate'].median():.1%}; "
             f"{(df['Savings_Rate'] > 0).mean():.0%} of households positive")

ax = axes[1, 0]
zero = (df[SHARE_COLS] == 0).mean().sort_values(ascending=False)
viz.barh(ax, [c.replace("_Share", "").replace("_", " ") for c in zero.index], list(zero.values),
         colors=[viz.BLUE if v > 0.5 else viz.CONTEXT for v in zero.values], value_fmt="{:.0%}",
         xlim=(0, 1.08))
ax.set_xlabel("share of households with zero spend")
ax.set_title(f"Zero spend: {(zero > 0.5).sum()} categories are absent for most households")

ax = axes[1, 1]
order = ["Metro_Urban", "Other_Urban", "Developed_Village", "Less_Developed_Village"]
r = by_area.reindex(order)
x = np.arange(4)
ax.bar(x - 0.18, r["rate"], width=0.34, color=viz.BLUE, label="unweighted")
ax.bar(x + 0.18, r["weighted"], width=0.34, color=viz.ORANGE, label="survey-weighted")
ax.set_xticks(x, [o.replace("_", "\n") for o in order])
viz.strip(ax, "y")
ax.set_ylabel("share meeting the 20% benchmark")
viz.legend(ax, loc="upper right")
ax.set_title(f"Goal attainment by area type (overall {rate:.1%}, weighted {rate_w:.1%})")
plt.savefig(C.RESULTS / "phase1_eda.png")
plt.show()